In [ ]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt

%matplotlib inline
%matplotlib notebook

## 1.Load file.

In [ ]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [ ]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

## 2. Unpaved ###

In [ ]:
iters = np.shape(P_atm)[0] # total timestep.

In [ ]:
UP_measure = 0 # we do not consider 'measure' for the time being.

### 2.1 Assumptions:

1. __Area of paved areas__: Unpaved area has interactions with paved roof, closed paved and open paved. The inflow from paved areas to unpaved area is dependent on the area of three paved areas, because the depth should be convert to volume (depth * area) first and then is converted to the depth increase in unpaved area. In order to build up the unpaved unit, several conditions are chosen: 

    a. only paved roof.
    
    b. three paved areas in together.
    
2. __Note that Only when discfrac is not equal to 0, then R_up is not zero.__ 

3. __So an implicit basic assumption__ here is that: If discfrac = 0, then all the water on the paved area will first enter the sewer system and then generate SO (overflow on the ground) if any, and no R_up is generated even given a very large rainfall.  If discfrac != 0, then there could a fraction of water going to unpaved area through R_up flux. 

4. __In order to validate__, for each case of condition, at leaset two sets of coefficient should be to tested.

### 2.2 Only paved roof case:

#### 2.2.1 parameter set 1(C1S1):
discfrac_pavedroof = 0.5, area_pavedroof = 1560 m^2, area_unpaved = 6855 m^2, stormfrac_pavedroof = 1.0

##### a. Input data preparation: #####
(a). __The results of R_up__ from paved roof module is:

In [ ]:
# remember to convert depth by the ratio of area

path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'

#case 1 setup 1
c1s1 = pd.read_csv(path + 'sol/' + 'Class_results_modified_forupbuildup_discfrac05.csv')
r_up_pr = c1s1[' R_up']
r_up_cp = np.zeros(60)
r_up_op = np.zeros(60)

pr_area = 1560
cp_area = 0
op_area = 0
up_area = 6855

(b). __The results of theta_rz__ from unsaturated module is (__used as input of mois_uz__ in unpaved module) copied from excel. Because unpaved module is connected with unsaturated zone and the theta_rz in unsaturated zone module is involved as input to the unpaved module. Hence an assumption is made here that we already know theta_rz and take them as input.

In [ ]:
theta_rz = [194.1,194.1,194.0943423,194.0887235,194.0831165,194.0775212,194.0743966,194.0764051,194.0732513,194.0714382,194.0656363,194.0598657
,194.0541071,194.0495991,194.0451737,194.0407571,194.0350326,194.0293264,194.0238028,194.0183326,194.0128737,194.0074262,194.0019903,193.9965657
,193.9911525,193.9857507,193.9803603,193.9749812,193.9696133,193.9642568,193.9589114,193.9535678,193.9481028,193.9426498,193.9372083,193.9317782
,193.9263595,193.9209522,193.9155562,193.9101715,193.9112873,193.9072155,193.9019557,193.8967411,193.8928924,193.8876902,193.8825053,193.8773313
,193.8735443,193.8683824,193.864614,193.8676569,193.8721982,193.8767236,193.8812403,193.8856785,193.8891301,193.8925658,193.8959949
,193.8994173]

(C). __Parameter set__ infilcap_up = 48 mm/d, mois_uz_max = 249.2, k_sat_uz = 67.9 Note that these three values are depend on the soil type.

In [ ]:
infilcap_up = 48
mois_uz_max = 249.2 
k_sat_uz = 67.9 
intstorcap_up = 20 # predefined storage capacity on unpaved area.

# Open water area. Because there is a determine statement in "final_storage": 
# For now, runoff from unpaved area is assumed to flow to the open water area. 
# If no open water area is present the water cannot runoff and will be stored on the surface of the unpaved area.
ow_area = 300 

#### b. Build up the unpaved module using arrays and while loop to understand the structure.

__Note that__ 

It is always good to build up the module using arrays first to get a full understanding of how it is structured. Because it makes it easier for us to validate each solution and convert this straightforward algorithm to a Class later on. 

#### Array building to understand the structure:

In [ ]:
delta_t = 1 / 24

sum_r_up = np.zeros(iters)
init_stor_up = np.zeros(iters)
fin_stor_up = np.zeros(iters)
act_infilcap_up = np.zeros(iters)
tfac = np.zeros(iters)
e_atm_up = np.zeros(iters)
i_uz_up = np.zeros(iters)
r_ow_up = np.zeros(iters)

mois_uz = theta_rz


# inflow factor unpaved
# there is no measure ---> meas_inflow_area = meas_area = 0, then inflowfac_up = 1.
meas_inflow_aera,  meas_area = 0, 0
inflowfac_up = (up_area - (meas_inflow_aera - meas_area)) / up_area

t = 1 

while t <= iters - 1:
    
    # ΣR_up
    sum_r_up[t] = (r_up_pr[t] * pr_area + r_up_cp[t] * cp_area + r_up_op[t] * op_area ) / (up_area)
    
    # initial storage
    init_stor_up[t] = fin_stor_up[t-1] + P_atm[t] + sum_r_up[t]
    
    # actual infiltration capacity
    # Remember to ask questions about this actual infiltration capacity part, especially why is v1 + min(v1, v2). 
    act_infilcap_up[t] = min(delta_t * infilcap_up, mois_uz_max - mois_uz[t-1] + min(mois_uz_max - mois_uz[t-1], delta_t * k_sat_uz))

    # time factor
    if E_pot_OW[t] + act_infilcap_up[t] <= 0:
        tfac[t] = 0
    else: 
        tfac[t] = min(1, init_stor_up[t] / (E_pot_OW[t] + act_infilcap_up[t]))
    
    e_atm_up[t] = tfac[t] * E_pot_OW[t]
    
    i_uz_up[t] = tfac[t] * act_infilcap_up[t]
    
    # Final storage part
    # Final storage part is tricky and confusing. Make sure to ask about this part and recheck it.
    if ow_area == 0:
        fin_stor_up[t] = max(0, min(intstorcap_up + 
                                inflowfac_up * (init_stor_up[t] - e_atm_up[t] - i_uz_up[t] - intstorcap_up), 
                                    init_stor_up[t]- e_atm_up[t] - i_uz_up[t]))
    else:
        fin_stor_up[t] = max(0, min(intstorcap_up, init_stor_up[t] - e_atm_up[t] - i_uz_up[t]))
    
    # R_ow
    if ow_area == 0:
        r_ow_up[t] = 0 
    else:
        r_ow_up[t] = max(0, init_stor_up[t] - e_atm_up[t] - i_uz_up[t] - intstorcap_up)
    
    t += 1
    
#print(r_ow_up)
filename = 'Results_Unpaved_c1s1_arraybuild.csv'
np.savetxt('sol/' + filename, np.c_[sum_r_up, init_stor_up, act_infilcap_up, tfac, e_atm_up, i_uz_up, fin_stor_up, r_ow_up], fmt = "%.8f", delimiter=',', header = 'sum_R_up, init_stor, act_infilcap, time_factor, E_atm, I_uz, fin_stor, R_ow') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

#### Convert to Class Unpaved:

In [ ]:
class Unpaved:
    def __init__(self, init_stor, fin_stor, up_area, intstorcap_unpaved = 20,  infilcap_unpaved = 48, mois_uz_max = 249.2, k_sat_uz = 67.9, inflowfac = 1):
        
        # state
        self.init_stor = init_stor
        self.fin_stor = fin_stor
        
        # parameter
        self.up_area = up_area
        self.intstorcap = intstorcap_unpaved
        self.infilcap = infilcap_unpaved 
        self.mois_uz_max = mois_uz_max
        self.k_sat_uz = k_sat_uz
        self.inflowfac = inflowfac
    
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are current precipitation and evaporation.'
    
    def sol(self, p_atm, e_pot_ow, r_up_pr, r_up_cp, r_up_op, mois_uz, pr_area, cp_area, op_area, ow_area, delta_t = 1 / 24): 
        
        # ΣR_up
        sum_r_up = (r_up_pr * pr_area + r_up_cp * cp_area + r_up_op * op_area) / (self.up_area)
        
        # initial storage
        init_stor = self.fin_stor + p_atm + sum_r_up
        
        # actual infiltration capacity
        act_infilcap = min(delta_t * self.infilcap, self.mois_uz_max - mois_uz + min(self.mois_uz_max - mois_uz, delta_t * self.k_sat_uz))
        
        # time factor
        if e_pot_ow + act_infilcap <= 0:
            tfac = 0
        else: 
            tfac = min(1, init_stor / (e_pot_ow + act_infilcap))
            
        # evap
        e_atm = tfac * e_pot_ow
        
        # infiltration
        i_uz = tfac * act_infilcap
            
        # final storage
        if ow_area == 0:
            fin_stor = max(0, min(self.intstorcap + self.inflowfac * (init_stor - e_atm - i_uz - self.intstorcap), init_stor - e_atm - i_uz))

        else:
            fin_stor = max(0, min(self.intstorcap, init_stor - e_atm - i_uz))
            
        # R_ow
        if ow_area == 0:
            r_ow = 0 
        else:
            r_ow = max(0, init_stor - e_atm - i_uz - self.intstorcap)
            
        # update state
        self.init_stor = init_stor
        self.fin_stor = fin_stor
        
        return sum_r_up, init_stor, act_infilcap, tfac, e_atm, i_uz, fin_stor, r_ow

In [ ]:
t = 1

sum_r_up = [0]
init_stor = [0]
act_infilcap= [0] 
tfac = [0] 
e_atm = [0] 
i_uz = [0]
fin_stor= [0] 
r_ow = [0]

# Give initial interception storage.
init_stor_t0, fin_stor_t0 = 0, 0

# Specify the parameter or use the default setting.
m = Unpaved(init_stor_t0, fin_stor_t0, 6855, intstorcap_unpaved = 20,  infilcap_unpaved = 48, mois_uz_max = 249.2, k_sat_uz = 67.9, inflowfac = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t], r_up_pr[t], r_up_cp[t], r_up_op[t], theta_rz[t-1], 1560, 0, 0, 300, 1/24)
    
    sum_r_up.append(sol[0])
    init_stor.append(sol[1])
    act_infilcap.append(sol[2])
    tfac.append(sol[3])
    e_atm.append(sol[4])
    i_uz.append(sol[5])
    fin_stor.append(sol[6])
    r_ow.append(sol[7])
    
    # print('time step', t)
    t += 1
    
filename = 'Results_Unpaved_c1s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_r_up, init_stor, act_infilcap, tfac, e_atm, i_uz, fin_stor, r_ow], fmt = "%.8f", delimiter=',', header = 'sum_R_up, init_stor, act_infilcap, time_factor, E_atm, I_uz, fin_stor, R_ow') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

#### 2.2.1 parameter set 2(C1S2):
discfrac_pavedroof = 0.3, area_pavedroof = 1560 m^2, area_unpaved = 6855 m^2, stormfrac_pavedroof = 0.7

##### a. Input data preparation: #####
(a). __The results of R_up__ from paved roof module is:

In [ ]:
# remember to convert depth by the ratio of area

path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'

#case 1 setup 1
c1s2 = pd.read_csv(path + 'sol/' + 'Class_results_modified_foruppavedbuildup_discfrac03storm07.csv')
r_up_pr = c1s2[' R_up']
r_up_cp = np.zeros(60)
r_up_op = np.zeros(60)

pr_area = 1560
cp_area = 0
op_area = 0
up_area = 6855

(b). __The results of theta_rz__

In [ ]:
theta_rz_c1s2 = [194.1,194.1,194.0943423,194.0887235,194.0831165,194.0775212,194.0743966,194.0763116,194.0730517,194.0710889,194.0652885
,194.0595186,194.0537608,194.0492419,194.0447741,194.0403155,194.0345921,194.0288869,194.0233642,194.0178949,194.0124369,194.0069904
,194.0015553,193.9961317,193.9907194,193.9853185,193.979929,193.9745508,193.9691838,193.9638282,193.9584837,193.953141,193.9476768
,193.9422248,193.9367842,193.931355,193.9259372,193.9205307,193.9151356,193.9097518,193.9106895,193.9065728,193.9013147,193.8961014,193.8922077
,193.8870072,193.8818237,193.8766511,193.8728107,193.8676506,193.8638288,193.8665444,193.8710894,193.8756169,193.8801358,193.8845761,193.8880299,193.8914677,193.8948989
,193.8983234]

(C). __Parameter set__ infilcap_up = 48 mm/d, mois_uz_max = 249.2, k_sat_uz = 67.9 Note that these three values are depend on the soil type.

In [ ]:
infilcap_up = 48
mois_uz_max = 249.2 
k_sat_uz = 67.9 
intstorcap_up = 20 
ow_area = 300 

In [ ]:
t = 1

sum_r_up = [0]
init_stor = [0]
act_infilcap= [0] 
tfac = [0] 
e_atm = [0] 
i_uz = [0]
fin_stor= [0] 
r_ow = [0]

# Give initial interception storage.
init_stor_t0, fin_stor_t0 = 0, 0

# Specify the parameter or use the default setting.
m = Unpaved(init_stor_t0, fin_stor_t0, 6855, intstorcap_unpaved = 20,  infilcap_unpaved = 48, mois_uz_max = 249.2, k_sat_uz = 67.9, inflowfac = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t], r_up_pr[t], r_up_cp[t], r_up_op[t], theta_rz_c1s2[t-1], 1560, 0, 0, 300, 1/24)
    
    sum_r_up.append(sol[0])
    init_stor.append(sol[1])
    act_infilcap.append(sol[2])
    tfac.append(sol[3])
    e_atm.append(sol[4])
    i_uz.append(sol[5])
    fin_stor.append(sol[6])
    r_ow.append(sol[7])
    
    # print('time step', t)
    t += 1
    
filename = 'Results_Unpaved_c1s2.csv'
np.savetxt('sol/' + filename, np.c_[sum_r_up, init_stor, act_infilcap, tfac, e_atm, i_uz, fin_stor, r_ow], fmt = "%.8f", delimiter=',', header = 'sum_R_up, init_stor, act_infilcap, time_factor, E_atm, I_uz, fin_stor, R_ow') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have  been validated.')

### 2.2 three paved area in together case (paved roof, closed paved, open paved):

#### 2.2.1 parameter set 1(C2S1):
1. discfrac_pavedroof = 0.35, area_pavedroof = 3000 m^2, stormfrac_pavedroof = 0.75
2. discfrac_closedpaved = 0.285, area_closedpaved = 1000 m^2, stormfrac_closedpaved = 0.75
3. discfrac_openpaved = 0.55, area_openpaved = 1500 m^2, stormfrac_openpaved 0.75
4. area_unpaved = 3500
5. area_ow = 1000

##### a. Input data preparation: #####
(a). __The results of R_up__ from paved roof module is:

In [ ]:
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'

#case 2 setup 1
c2s1_pr = pd.read_csv(path + 'sol/' + 'Class_results_modified_foruppavedbuildup_c2s1_discfrac035storm075.csv')
r_up_pr = c2s1_pr[' R_up']

c2s1_cp = pd.read_csv(path + 'sol/' + 'Results_closedpaved_forunpavedbuildup_c2s1_discfrac0285storm075.csv')
r_up_cp = c2s1_cp[' R_up']

c2s1_op = pd.read_csv(path + 'sol/' + 'Results_OpenPaved_forunpavedbuildup_c2s1_disfrac055stormfrac075.csv')
r_up_op = c2s1_op[' R_up']

pr_area = 3000
cp_area = 1000
op_area = 1500
up_area = 3500
ow_area = 1000

(b). __The results of theta_rz__ (directly get from excel):

In [ ]:
theta_rz_c2s1 = [194.1,194.1,194.0943423,194.0887152,194.0831,194.0774966,194.0736344,194.0741188,194.0710795,194.0693143,194.0635796
,194.0578702,194.0521728,194.0474616,194.0430541,194.0386548,194.0329938,194.0273488,194.0218339,194.0163598,194.0108971,194.0054458
,194.0000061,193.9945778,193.9891609,193.9837554,193.9783613,193.9729785,193.967607,193.9622469,193.956898,193.9515537,193.9461288
,193.9407156,193.9353138,193.9299233,193.9245441,193.9191763,193.9138197,193.9084743,193.9090866,193.9050441,193.8997917,193.894574
,193.8906898,193.8854867,193.8802989,193.8751219,193.8713592,193.8661964,193.8624521,193.8640998,193.8658631,193.8676225,193.8693785
,193.872348,193.8733648,193.8743272,193.8752879,193.8762467]

In [ ]:
t = 1

sum_r_up = [0]
init_stor = [0]
act_infilcap= [0] 
tfac = [0] 
e_atm = [0] 
i_uz = [0]
fin_stor= [0] 
r_ow = [0]

# Give initial interception storage.
init_stor_t0, fin_stor_t0 = 0, 0

# Specify the parameter or use the default setting.
m = Unpaved(init_stor_t0, fin_stor_t0, 3500, intstorcap_unpaved = 20,  infilcap_unpaved = 48, mois_uz_max = 249.2, k_sat_uz = 67.9, inflowfac = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t], r_up_pr[t], r_up_cp[t], r_up_op[t], theta_rz_c2s1[t-1], 3000, 1000, 1500, 1000, 1/24)
    
    sum_r_up.append(sol[0])
    init_stor.append(sol[1])
    act_infilcap.append(sol[2])
    tfac.append(sol[3])
    e_atm.append(sol[4])
    i_uz.append(sol[5])
    fin_stor.append(sol[6])
    r_ow.append(sol[7])
    
    # print('time step', t)
    t += 1
    
filename = 'Results_Unpaved_c2s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_r_up, init_stor, act_infilcap, tfac, e_atm, i_uz, fin_stor, r_ow], fmt = "%.8f", delimiter=',', header = 'sum_R_up, init_stor, act_infilcap, time_factor, E_atm, I_uz, fin_stor, R_ow') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have  been validated.')

### Conclusion.


1. The unpaved module has been created and validated with three cases (2 sets for only paved roof, 1 set for all three paved areas).
2. It is good to know further about the final storage part as well as the actual infiltration capacity part as I do not quite understand it even though I have programmed it.
